In [ ]:
import itertools
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.surface_code.teleportation import TeleportationSurgery
from library.qubit_array import QubitArray

In [ ]:
qubits = QubitArray(dimensions=(9, 5))
circuitry = Circuitry(qubits=qubits)
basis = Pauli.Z

surgery = TeleportationSurgery(qubits, distance=3, anchor=(1, 1))
surgery.append_movement(circuitry, prepare=basis, measure=basis)

for qubit, records in qubits.measurements_qubit.items():
    if len(records) > 1:
        patches = {key for key, _ in itertools.groupby(records, key=lambda mr: mr[0])}

        if len(patches) == 1:
            # Purely in SOURCE or TARGET patches
            stabilizer = records[0].split(":")[-1]
            if stabilizer.startswith(basis.name):
                circuitry.annotate_detector(records[0])

            for rounds in itertools.pairwise(records):
                circuitry.annotate_detector(*rounds)
        elif len(patches) == 2:  # Crossing SOURCE&MERGER or MERGER&TARGET patches
            stabilizer = records[0].split(":")[-1]
            if stabilizer.startswith(basis.name):
                circuitry.annotate_detector(records[0])

            for sub_records in [records[:3], records[3:]]:
                for rounds in itertools.pairwise(sub_records):
                    circuitry.annotate_detector(*rounds)

circuitry.annotate_detector(
    "SRC:M1:R2:Z0", "SRC:M1:R2:D0", "SRC:M1:R2:D1", "SRC:M1:R2:D3", "SRC:M1:R2:D4"
)
circuitry.annotate_detector("SRC:M1:R2:Z2", "SRC:M1:R2:D1", "SRC:M1:R2:D2")
circuitry.annotate_detector(
    "SRC:M1:R2:Z3", "SRC:M1:R2:D4", "SRC:M1:R2:D5", "SRC:M1:R2:D7", "SRC:M1:R2:D8"
)

circuitry.annotate_detector("SRC:M0:R2:Z1", "JCT:M1:R0:Z0")

circuitry.annotate_detector(
    "JCT:M1:R2:Z0", "SRC:M1:R2:D6", "SRC:M1:R2:D7", "JCT:M1:R2:D3", "JCT:M1:R2:D4"
)
circuitry.annotate_detector(
    "JCT:M1:R2:Z3", "JCT:M1:R2:D4", "JCT:M1:R2:D5", "TGT:M2:R0:Z2"
)

circuitry.annotate_detector(
    "TGT:M2:R2:Z0", "TGT:M2:R2:D0", "TGT:M2:R2:D1", "TGT:M2:R2:D3", "TGT:M2:R2:D4"
)
circuitry.annotate_detector("TGT:M2:R2:Z1", "TGT:M2:R2:D6", "TGT:M2:R2:D7")
circuitry.annotate_detector("TGT:M2:R2:Z2", "TGT:M2:R2:D1", "TGT:M2:R2:D2")
circuitry.annotate_detector(
    "TGT:M2:R2:Z3", "TGT:M2:R2:D4", "TGT:M2:R2:D5", "TGT:M2:R2:D7", "TGT:M2:R2:D8"
)

surgery.annotate_observable(
    circuitry,
    0,
    *[
        "SRC:M1:R2:D1",
        "SRC:M1:R2:D4",
        "SRC:M1:R2:D7",
        "JCT:M1:R2:D4",
        "TGT:M2:R2:D1",
        "TGT:M2:R2:D4",
        "TGT:M2:R2:D7",
    ],
)

missing = circuitry.missing_detectors()
print(f"Missing detectors : {len(missing)}")

display(Markdown(f"[Open in Crumble]({circuitry.to_crumble_url()})"))

In [ ]:
for label, _ in qubits.measurements_index.items():
    print(f"Label {label} : {qubits.retrieve_measurement(label)}")